# Ask Finance — Interactive Demo

This notebook walks through the **end-to-end capability** of the Ask Finance agent:
1. Load mock SAP/HFM + project data
2. Switch between different user personas (Group CFO, BU GM, Regional BP, Analyst)
3. Ask the same question as different users — watch RBAC kick in
4. Inspect tool-call traces and data citations
5. Generate chart + Excel outputs

> **Note** — the agent uses Claude (Anthropic) if `ANTHROPIC_API_KEY` is set.
> Otherwise it falls back to a deterministic **mock planner** so reviewers can
> still run the full demo without an API key.

In [ ]:
import sys, os
sys.path.insert(0, '..')

from src.rbac import get_context, load_users
from src.agent import ask
from src.tools import run_tool
from src.data_loader import actuals_for, projects_for, data_catalog

print('LLM provider:', 'anthropic (live)' if os.getenv('ANTHROPIC_API_KEY') else 'mock (deterministic)')

## 1. The simulated user directory
Five users, five roles. Each one has a different slice of the data.

In [ ]:
import pandas as pd
pd.DataFrame(load_users())

## 2. Data catalog per role
`data_catalog()` returns exactly what a user is allowed to see. This is what the LLM also sees in its system prompt.

In [ ]:
for uid in ['u001', 'u002', 'u003', 'u004', 'u005']:
    c = get_context(uid)
    cat = data_catalog(c)
    print(f'\n=== {c.name}  |  {c.role}  |  {c.scope_description()} ===')
    print(f'   BUs visible     : {cat["actuals"]["business_units"]}')
    print(f'   regions visible : {cat["actuals"]["regions"]}')
    print(f'   projects visible: {cat["projects"]["projects"]}')
    print(f'   budget access   : {cat["budget_access"]}')

## 3. End-to-end query — Group CFO asks for EBIT margin trend

In [ ]:
ctx = get_context('u001')   # Group CFO
trace = ask(ctx, "What's our EBIT margin trend for FY2023, FY2024 and FY2025?")

from IPython.display import Markdown
Markdown(trace.final_text)

In [ ]:
# Inspect the tool-call trace → full explainability
for tc in trace.tool_calls:
    print('→', tc['name'], '(', tc['input'], ')')
    if 'series' in tc['result']:
        for p in tc['result']['series']:
            print('  ', p)

## 4. Same question, different users → RBAC changes the answer
Notice the numbers shrink as scope narrows.

In [ ]:
Q = 'Summarize the FY2024 P&L highlights for my region.'
for uid in ['u001', 'u002', 'u003', 'u004']:
    c = get_context(uid)
    t = ask(c, Q)
    print(f'\n━━━ {c.name}  ({c.role} — {c.scope_description()}) ━━━')
    print(t.final_text)

## 5. RBAC denial example — Analyst asks for budget data

In [ ]:
c = get_context('u005')   # Analyst — no budget access
t = ask(c, 'Show me Opex variance vs budget for Q1 FY2024')
print(t.final_text)

## 6. Project ROI — Project Orion over 3 years
Only users whose scope intersects Orion's (Electronics + APAC) can see it.

In [ ]:
for uid in ['u001', 'u002', 'u003', 'u004']:
    c = get_context(uid)
    t = ask(c, 'Show me the ROI trend of Project Orion over the last 3 years.')
    print(f'\n━━━ {c.name}  ({c.role}) ━━━')
    print(t.final_text)

## 7. Chart generation

In [ ]:
ctx = get_context('u001')
# Get EBIT margin trend data
trend = run_tool('get_metric_trend', ctx, {
    'metric': 'EBIT Margin %',
    'fiscal_years': ['FY2023', 'FY2024', 'FY2025'],
})
# Render chart
series = [{'x': p['fiscal_year'], 'y': p['value']} for p in trend['series']]
out = run_tool('generate_chart', ctx, {
    'filename': 'ebit_margin_trend.png',
    'title':    'EBIT Margin % — Group Total',
    'x_label':  'Fiscal Year',
    'y_label':  'EBIT Margin %',
    'series':   series,
    'chart_type': 'bar',
})
print(out)
from IPython.display import Image
Image(out['generated_file'])

## 8. Excel export

In [ ]:
records = projects_for(ctx).to_dict(orient='records')
out = run_tool('generate_excel', ctx, {
    'filename': 'project_roi.xlsx',
    'title':    'Strategic Project ROI — Group View',
    'rows':     records,
})
out

## 9. Audit log
Every user query + tool invocation is appended to `outputs/audit_log.jsonl`.

In [ ]:
from pathlib import Path
log_path = Path('../outputs/audit_log.jsonl')
if log_path.exists():
    print(log_path.read_text().strip().splitlines()[-5][:300])